# Day 1 - ViT 복습

- status: implemented_toy_not_executed_real_model
- stage: VLM_DAY_1
- paper_ids: vit_2020
- dataset_ids: synthetic_toy, user_selected_nas_data
- seed: 42
- scope: educational implementation; real inference/training is opt-in

이 노트북은 다른 사용자 파일, 공유 환경, checkpoint를 자동으로 변경하지 않는다. 실제 데이터는
`/nas/datahub/min` 아래 사용자가 지정한 경로만 읽는다.

## 1. Learning question

이미지를 token sequence로 바꾸면 Transformer가 무엇을 보게 되는가? patch embedding, positional information, self-attention의 shape를 손으로 확인한다.

## 2. Background theory

`image [B,C,H,W] -> non-overlapping patches [B,N,P*P*C] -> linear embedding [B,N,D] -> position -> Transformer`. `N=(H/P)*(W/P)`이며 full self-attention의 score 행렬은 head마다 `[N,N]`이다. 해상도를 2배로 하면 token은 약 4배, attention 요소는 약 16배가 된다.

## 3. Paper connection

ViT는 CNN의 spatial inductive bias를 줄이고 image patch를 NLP token처럼 처리했다. 여기서는 pretrained accuracy가 아니라 patchification과 attention shape를 재현한다.

## 4. Input/output and shapes

입력은 HWC toy image, 출력 patch는 `[N,P*P*C]`다. 실제 구현은 patch projection 뒤 `[B,N,D]`가 되며 class token 또는 pooled representation이 global task에 사용된다.

In [ ]:
from pathlib import Path
import sys
import numpy as np

current = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (current, *current.parents) if (p / "pyproject.toml").is_file()),
    Path("/nas/home/mhlee/vlm-foundation-7days"),
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("project:", PROJECT_ROOT)
print("numpy:", np.__version__)

## 5. Minimal implementation

작은 8x8 image를 4x4 patch 네 개로 분해하고 한 attention head의 scaled dot-product score를 만든다.

In [ ]:
from vlm_foundation.vit import attention_scores, patchify

image = np.arange(8 * 8).reshape(8, 8, 1)
patches = patchify(image, patch_size=4)
projection = np.eye(patches.shape[1], 4)
tokens = patches @ projection
scores = attention_scores(tokens, np.eye(4), np.eye(4))
print("patches:", patches.shape)
print("tokens:", tokens.shape)
print("attention scores:", scores.shape)

## 6. Visualization sanity check

patch 순서를 좌상단에서 우하단 raster order로 확인한다. patch transpose 순서가 틀리면 내용은 유지돼 보여도 spatial position이 뒤섞인다.

In [ ]:
print("첫 patch 4x4:\n", patches[0].reshape(4, 4))
print("두 번째 patch 4x4:\n", patches[1].reshape(4, 4))

## 7. Experiment

patch size 2와 4에서 token 수와 attention matrix 크기를 비교한다.

In [ ]:
from vlm_foundation.vit import patch_grid
for patch_size in [2, 4]:
    gh, gw = patch_grid(8, 8, patch_size)
    n = gh * gw
    print(f"patch={patch_size}: tokens={n}, attention elements={n*n}")

## 8. Metrics

이날의 metric은 shape invariant와 patch reconstruction 오차다. 모델 accuracy는 다루지 않는다.

## 9. Interpretation

ViT가 보는 기본 단위는 pixel 하나가 아니라 patch token이다. 작은 물체가 patch보다 작으면 초기 표현부터 정보가 뭉개질 수 있다.

## 10. Failure cases

채널 순서, 정규화, H/W transpose, divisibility, positional embedding interpolation 오류를 확인한다.

## 11. Real-service implications

고해상도 OCR과 grounding에서는 patch 수가 곧 memory와 latency로 이어진다. 입력 해상도 정책은 모델 밖의 사소한 전처리가 아니다.

## 12. Review questions

1. patch size를 절반으로 하면 token 수는 어떻게 변하는가?
2. position 정보가 없으면 어떤 구분이 어려운가?
3. `[N,N]` attention이 해상도에 민감한 이유는?
4. CLS pooling과 dense visual token의 용도 차이는?
5. VLM에서 global feature 하나만으로 부족한 작업은 무엇인가?